# Adaptive Optimization Algorithms — Reproducible Experiments

**Research context:** BIT-SIPAR Summer Internship Program for Advanced Research, BIT Mesra  
**Topic:** Adaptive optimization methods for minimizing a least-squares objective on the MA201 dataset.

This notebook provides a consolidated, reproducible implementation of:
1. Fixed-step Gradient Descent
2. Exact Line Search for the least-squares objective
3. Bisection Line Search (based on the directional derivative)
4. Backtracking Line Search (Armijo sufficient-decrease condition)
5. A simple adaptive step-size hedging heuristic

## Before running
Place the authorized MA201 CSV file at `data/MA201 - Sheet1.csv`, relative to this notebook. The dataset is **not included**. Do not publish the data unless you have permission.

**Important:** The simple adaptive step-size hedging routine below follows the heuristic used in the uploaded notebook (increase step size after improvement, reduce it after failure). It should not be described as a full implementation of the Silver Stepsize Schedule from the cited theoretical literature.


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data") / "MA201 - Sheet1.csv"
FEATURE_COLUMNS = ["Q1", "Q2", "Asgn", "MT", "ET"]
TARGET_COLUMN = "y"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Create a 'data' folder next to this notebook "
        "and place the permitted MA201 CSV there."
    )

df = pd.read_csv(DATA_PATH)
required = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_cols = [col for col in required if col not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values in required columns:")
display(df[required].isna().sum().to_frame("missing_count"))
print("\nExact duplicate rows:", int(df.duplicated().sum()))


In [ ]:
# Prepare complete numeric observations for the specified features/target.
# Rows missing any required value are excluded; no values are imputed.
data = df[required].apply(pd.to_numeric, errors="coerce").dropna().copy()
if data.empty:
    raise ValueError("No complete numeric observations remain after selecting required columns.")

X_raw = data[FEATURE_COLUMNS].to_numpy(dtype=float)
y = data[TARGET_COLUMN].to_numpy(dtype=float).reshape(-1, 1)

# Standardize features; protect against zero-variance columns.
mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
sigma[sigma == 0] = 1.0
X_scaled = (X_raw - mu) / sigma

# Add intercept column (not standardized).
X = np.column_stack([np.ones(X_scaled.shape[0]), X_scaled])
m, n = X.shape
theta0 = np.zeros((n, 1))

print(f"Rows used: {m}; parameters including intercept: {n}")
print("Design matrix shape:", X.shape, "| target shape:", y.shape)


In [ ]:
def predict(X, theta):
    return X @ theta

def mse_cost(X, y, theta):
    residual = predict(X, theta) - y
    return float(np.mean(residual ** 2))

def objective(X, y, theta):
    # Half mean squared error, matching the report's cost convention.
    residual = predict(X, theta) - y
    return float(np.sum(residual ** 2) / (2 * len(y)))

def gradient(X, y, theta):
    return (X.T @ (X @ theta - y)) / len(y)

def exact_step(X, g):
    # Exact minimizer along d = -g for the quadratic least-squares objective.
    Xg = X @ g
    denom = float((Xg.T @ Xg) / len(X))
    numer = float(g.T @ g)
    if denom <= 1e-15:
        return 0.0
    return numer / denom

def run_gradient_descent(X, y, theta_init, max_iter=1000, tol=1e-8, alpha=0.05):
    theta = theta_init.copy()
    history = []
    start = time.perf_counter()
    for _ in range(max_iter):
        g = gradient(X, y, theta)
        if np.linalg.norm(g) <= tol:
            break
        theta -= alpha * g
        history.append(objective(X, y, theta))
    return theta, history, time.perf_counter() - start

def run_exact_line_search(X, y, theta_init, max_iter=1000, tol=1e-8):
    theta = theta_init.copy()
    history = []
    start = time.perf_counter()
    for _ in range(max_iter):
        g = gradient(X, y, theta)
        if np.linalg.norm(g) <= tol:
            break
        alpha = exact_step(X, g)
        if alpha <= 0:
            break
        theta -= alpha * g
        history.append(objective(X, y, theta))
    return theta, history, time.perf_counter() - start

def run_bisection_line_search(X, y, theta_init, max_iter=1000, tol=1e-8,
                              alpha_low=0.0, alpha_high=1.0, alpha_tol=1e-8):
    # For the convex quadratic along d=-g, bisect the directional derivative
    # until it changes sign or the step-size interval is sufficiently small.
    theta = theta_init.copy()
    history = []
    start = time.perf_counter()
    for _ in range(max_iter):
        g = gradient(X, y, theta)
        if np.linalg.norm(g) <= tol:
            break
        d = -g
        lo, hi = alpha_low, alpha_high
        def directional_derivative(a):
            return float((gradient(X, y, theta + a*d).T @ d).item())
        dlo, dhi = directional_derivative(lo), directional_derivative(hi)
        expansions = 0
        while dhi < 0 and expansions < 60:
            hi *= 2.0
            dhi = directional_derivative(hi)
            expansions += 1
        if dlo >= 0:
            alpha = 0.0
        elif dhi <= 0:
            alpha = hi
        else:
            while hi - lo > alpha_tol:
                mid = (lo + hi) / 2.0
                dm = directional_derivative(mid)
                if dm < 0:
                    lo = mid
                else:
                    hi = mid
            alpha = (lo + hi) / 2.0
        if alpha <= 0:
            break
        theta = theta + alpha*d
        history.append(objective(X, y, theta))
    return theta, history, time.perf_counter() - start

def run_backtracking(X, y, theta_init, max_iter=1000, tol=1e-8,
                     alpha0=1.0, rho=0.5, c=1e-4):
    theta = theta_init.copy()
    history = []
    start = time.perf_counter()
    for _ in range(max_iter):
        g = gradient(X, y, theta)
        if np.linalg.norm(g) <= tol:
            break
        f0 = objective(X, y, theta)
        alpha = alpha0
        while objective(X, y, theta - alpha*g) > f0 - c*alpha*float((g.T @ g).item()):
            alpha *= rho
            if alpha < 1e-14:
                break
        if alpha < 1e-14:
            break
        theta -= alpha*g
        history.append(objective(X, y, theta))
    return theta, history, time.perf_counter() - start

def run_step_size_hedging(X, y, theta_init, max_iter=1000, tol=1e-8,
                          alpha0=0.1, growth=1.05, shrink=0.5):
    # Heuristic adaptive-step routine matching the uploaded notebook's approach.
    theta = theta_init.copy()
    alpha = alpha0
    history = []
    start = time.perf_counter()
    current = objective(X, y, theta)
    for _ in range(max_iter):
        g = gradient(X, y, theta)
        if np.linalg.norm(g) <= tol:
            break
        candidate = theta - alpha*g
        candidate_cost = objective(X, y, candidate)
        if candidate_cost < current:
            theta = candidate
            current = candidate_cost
            alpha *= growth
            history.append(current)
        else:
            alpha *= shrink
            if alpha < 1e-14:
                break
    return theta, history, time.perf_counter() - start


In [ ]:
algorithms = {
    "Gradient Descent": lambda: run_gradient_descent(X, y, theta0, alpha=0.05),
    "Exact Line Search": lambda: run_exact_line_search(X, y, theta0),
    "Bisection Line Search": lambda: run_bisection_line_search(X, y, theta0),
    "Backtracking Line Search": lambda: run_backtracking(X, y, theta0),
    "Step Size Hedging (heuristic)": lambda: run_step_size_hedging(X, y, theta0),
}

results = {}
for name, run in algorithms.items():
    theta_hat, history, runtime = run()
    results[name] = {
        "theta": theta_hat,
        "history": history,
        "iterations": len(history),
        "final_half_MSE": objective(X, y, theta_hat),
        "final_MSE": mse_cost(X, y, theta_hat),
        "runtime_seconds": runtime,
    }

summary = pd.DataFrame([
    {
        "Algorithm": name,
        "Iterations": result["iterations"],
        "Final half-MSE objective": result["final_half_MSE"],
        "Final MSE": result["final_MSE"],
        "Runtime (seconds)": result["runtime_seconds"],
    }
    for name, result in results.items()
]).sort_values("Final half-MSE objective")

display(summary)


In [ ]:
plt.figure(figsize=(9, 5))
for name, result in results.items():
    if result["history"]:
        plt.plot(result["history"], label=name)
plt.xlabel("Accepted iterations")
plt.ylabel("Half-MSE objective")
plt.title("Optimization convergence comparison")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Interpretation and reproducibility notes

- Runtime depends on hardware, Python/NumPy versions, and current system load; do not treat a single run as a universal benchmark.
- The number of accepted iterations depends on the stopping criterion and implementation details.
- The `Step Size Hedging (heuristic)` function implements the simple increase-on-success / shrink-on-failure routine from the uploaded notebook. It is **not** the same as a theoretically derived Silver Stepsize Schedule.
- Results in the internship report should be cited as report-reported results unless reproduced with this notebook and the same authorized dataset.
- The dataset is deliberately excluded from this repository template. Add it locally in `data/` only if permitted, and do not commit private or restricted data.
